In [1]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image
import cv2
import os
import random
from tensorflow.keras import layers


/workspace/tf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-02 13:32:30.995595: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764682351.010942   73189 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764682351.015699   73189 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764682351.029161   73189 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:17

In [2]:
class GroupNorm(layers.Layer):
    def __init__(self, groups=8, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.groups = groups
        self.epsilon = epsilon

    def build(self, input_shape):
        C = int(input_shape[-1])
        self.groups = min(self.groups, C)
        self.gamma = self.add_weight(name="gamma", shape=(C,), initializer="ones")
        self.beta  = self.add_weight(name="beta",  shape=(C,), initializer="zeros")

    def call(self, x):
        N, H, W, C = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], tf.shape(x)[3]
        G = self.groups

        x = tf.reshape(x, [N, H, W, G, C // G])
        mean, var = tf.nn.moments(x, axes=[1,2,4], keepdims=True)
        x = (x - mean) / tf.sqrt(var + self.epsilon)
        x = tf.reshape(x, [N, H, W, C])

        return x * self.gamma + self.beta


class TopKPooling(layers.Layer):
    def __init__(self, k=3, **kwargs):
        super().__init__(**kwargs)
        self.k = k

    def call(self, x):
        scores = tf.reduce_max(x, axis=-1)
        topk_idx = tf.argsort(scores, direction="DESCENDING")[:, :self.k]

        batch = tf.range(tf.shape(x)[0])[:, None]
        batch = tf.tile(batch, [1, self.k])

        gather_idx = tf.stack([batch, topk_idx], axis=-1)
        selected = tf.gather_nd(x, gather_idx)

        return tf.reduce_mean(selected, axis=1)


In [3]:
# Custom CNN 
custom_model = tf.keras.models.load_model(
    "/workspace/New Models/best_cnn_model_clahe_2.keras"
)

# VGG16 model
vgg_model = tf.keras.models.load_model(
    "/workspace/New Models/best_vgg16_clahe.keras"
)

# Patch-level model
patch_model = tf.keras.models.load_model(
    "/workspace/New Models/best_patch_model.keras",
    custom_objects={
        "GroupNorm": GroupNorm,
        "TopKPooling": TopKPooling,
        "TimeDistributed": layers.TimeDistributed,
    },
    compile=False
)

IMG_SIZE = 224
PATCH_SIZE = 56
NUM_PATCHES = 16
CLASS_NAMES = ["CNV", "DME", "DRUSEN", "NORMAL"]


I0000 00:00:1764682353.829013   73189 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79196 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:44:00.0, compute capability: 8.0


In [4]:
clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(4,4))

def apply_clahe_np(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    L, A, B = cv2.split(lab)
    L2 = clahe.apply(L)
    merged = cv2.merge((L2, A, B))
    out = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)
    return out


In [5]:
def preprocess_single_image(img_pil):
    img = np.array(img_pil)

    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    img = apply_clahe_np(img).astype("float32")
    img = img / 255.0

    return np.expand_dims(img, axis=0).astype("float32")


In [6]:
def extract_16_patches(img_rgb):

    if img_rgb.ndim == 2:
        img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_GRAY2RGB)

    img = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    img = apply_clahe_np(img).astype("float32")
    img = img / 255.0

    patches = []

    for i in range(0, IMG_SIZE, PATCH_SIZE):
        for j in range(0, IMG_SIZE, PATCH_SIZE):
            patch = img[i:i+PATCH_SIZE, j:j+PATCH_SIZE, :]
            patches.append(patch.astype("float32"))

    patches = np.stack(patches, axis=0).astype("float32")  # (16, 56, 56, 3)
    return patches


In [7]:
def predict_custom(img):
    x = preprocess_single_image(img)
    return custom_model.predict(x, verbose=0)[0]


def predict_vgg(img):
    x = preprocess_single_image(img)
    return vgg_model.predict(x, verbose=0)[0]


def predict_patch(img):
    arr = np.array(img)

    if arr.ndim == 2:
        arr = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
    else:
        arr = cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)

    patches = extract_16_patches(arr)

    patches = np.expand_dims(patches, axis=0).astype("float32")  # (1, 16, 56, 56, 3)

    probs = patch_model.predict(patches, verbose=0)[0]
    return probs


In [8]:
def predict_all(img):
    try:
        results = {}

        # Custom CNN
        p1 = predict_custom(img)
        results["Custom CNN"] = {
            "Prediction": CLASS_NAMES[np.argmax(p1)],
            "Confidence": float(np.max(p1)),
        }

        # VGG
        p2 = predict_vgg(img)
        results["VGG16"] = {
            "Prediction": CLASS_NAMES[np.argmax(p2)],
            "Confidence": float(np.max(p2)),
        }

        # Patch Model
        p3 = predict_patch(img)
        results["Patch Model"] = {
            "Prediction": CLASS_NAMES[np.argmax(p3)],
            "Confidence": float(np.max(p3)),
        }

        return results

    except Exception as e:
        import traceback
        traceback.print_exc()
        return {"error": str(e)}


In [9]:
with gr.Blocks(title="Retinal OCT Disease Classifier") as demo:

    gr.Markdown("# Retinal OCT Disease Classifier")
    gr.Markdown("Custom CNN + VGG16 + Patch MIL Model")

    with gr.Row():
        with gr.Column():
            img_in = gr.Image(type="pil", label="Upload OCT Image")
            btn = gr.Button("Classify")

        with gr.Column():
            out_box = gr.JSON(label="Predictions")

    btn.click(
        fn=predict_all,
        inputs=img_in,
        outputs=out_box
    )

demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://29c1960442331f704f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
